In [ ]:
import cv2
import matplotlib.pyplot as plt
import ultralytics
import numpy as np
import gc

ultralytics.checks()

# source_video_path = '../../акбарс_цска_короткое.mp4'
source_video_path = '../../тест_минута.mp4'

gc.collect()

In [ ]:
# Загрузка моделей для детекции поля и игроков
field_detection_model = ultralytics.YOLO('../../weights/field_detection_nikita.pt')
player_detection_model = ultralytics.YOLO('../../weights/player_detection_nikita.pt')

# Предсказание местоположения поля на видео
field_detection_results = field_detection_model.predict(
    source=source_video_path,
    conf=0.3,
    iou=0.5,
    device='mps',
    save=True,

    stream=True,
)

gc.collect()

# Предсказание местоположения игроков на видео
player_detection_results = player_detection_model.predict(
    source=source_video_path,
    conf=0.3,
    device='mps',
    # save=True,

    stream=True,
)

In [ ]:
# real_field_coordinates = {
#     'left_vorota': [0.06, 0.5],
#     'left_top_red_circle': [0.17, 0.246],
#     'left_bottom_red_circle': [0.17, 0.754],
#     'left_top_red_circle_upper_point': [0.235, 0.17],
#     'left_bottom_red_circle_upper_point': [0.234, 0.66],

#     'center_left_top_red_circle': [0.375, 0.25],
#     'center_left_bottom_red_circle': [0.375, 0.75],

#     'center_right_top_red_circle': [0.593, 0.25],
#     'center_right_bottom_red_circle': [0.593, 0.75],

#     'center': [0.495, 0.5],
    
#     'right_vorota': [0.940, 0.5],
#     'right_top_red_circle': [0.833, 0.25],
#     'right_bottom_red_circle': [0.833, 0.75],
#     'right_top_red_circle_upper_point': [0.76, 0.19],
#     'right_bottom_red_circle_upper_point': [0.763, 0.678],



#     'left_blue_line_up': [0.35, 0],
#     'right_blue_line_up': [0.62, 0.01],
#     'left_blue_line_bottom': [0.34, 0.96],
#     'right_blue_line_bottom': [0.63, 0.96],
# }

real_field_coordinates = {
    'left_vorota': [0.06, 0.5],
    'left_top_red_circle': [0.17, 0.246],
    'left_bottom_red_circle': [0.17, 0.754],
    'left_top_red_circle_upper_point': [0.235, 0.17],
    'left_bottom_red_circle_upper_point': [0.235, 0.674],

    'center_left_top_red_circle': [0.375, 0.25],
    'center_left_bottom_red_circle': [0.375, 0.75],

    'center_right_top_red_circle': [0.593, 0.25],
    'center_right_bottom_red_circle': [0.593, 0.75],

    'center': [0.495, 0.5],
    
    'right_vorota': [0.940, 0.5],
    'right_top_red_circle': [0.833, 0.25],
    'right_bottom_red_circle': [0.833, 0.75],
    'right_top_red_circle_upper_point': [0.76, 0.19],
    'right_bottom_red_circle_upper_point': [0.763, 0.678],



    'left_blue_line_up': [0.35, 0],
    'right_blue_line_up': [0.62, 0.01],
    'left_blue_line_bottom': [0.34, 0.96],
    'right_blue_line_bottom': [0.63, 0.96],
}


field_numbers = {
    'left_vorota': 1,
    'left_top_red_circle': 2,
    'left_bottom_red_circle': 3,
    'left_top_red_circle_upper_point': 12,
    'left_bottom_red_circle_upper_point': 11,

    'center_left_top_red_circle': 4,
    'center_left_bottom_red_circle': 5,

    'center_right_top_red_circle': 6,
    'center_right_bottom_red_circle': 7,

    'center': 19,

    'right_vorota': 10,
    'right_top_red_circle': 8,
    'right_bottom_red_circle': 9,
    'right_top_red_circle_upper_point': 18,
    'right_bottom_red_circle_upper_point': 17,

    'left_blue_line_up': 14,
    'right_blue_line_up': 15,

    'left_blue_line_bottom': 13,
    'right_blue_line_bottom': 16,
}

In [ ]:
all_player_coordinates = []
all_player_classes = []

for current_player_result in player_detection_results:
    all_player_coordinates.append(current_player_result.boxes.xywhn.to('cpu').numpy())
    all_player_classes.append(current_player_result.boxes.cls.to('cpu').numpy())

all_field_classes = []
all_field_boxes = []
all_field_masks = []
all_images = []

for current_field_result in field_detection_results:
    all_field_classes.append(current_field_result.boxes.cls.to('cpu').numpy())
    all_field_boxes.append(current_field_result.boxes.xywhn.to('cpu').numpy())
    
    if current_field_result.masks is not None:
        all_field_masks.append(current_field_result.masks.xyn)
    else:
        all_field_masks.append([])

    all_images.append(current_field_result.orig_img)

In [ ]:
field_image = cv2.imread('field_3.png')
field_image = cv2.resize(field_image, (400, int(400 / (675 / 348))))
field_image_height, field_image_width, _ = field_image.shape

class_to_id = {y:x for x,y in field_detection_model.names.items()}
class_to_id

In [ ]:
def find_center(center_boxes):
    areas = center_boxes[:, 2] * center_boxes[:, 3]
    center_boxes = center_boxes[areas > 0.01]

    if len(center_boxes):
        center_box = center_boxes[0]
        
        if center_box[0] + center_box[2] / 2 > 0.997:
            center_box[0] = (center_box[0] - center_box[2]/2) + center_box[3]/2 * 1.5
        elif center_box[0] - center_box[2] / 2 < 0.003:
            center_box[0] = (center_box[0] + center_box[2]/2) - center_box[3]/2 * 1.5
        
        return center_box

    return None


# Открытие видео и подготовка для записи
video_capture = cv2.VideoCapture(source_video_path)
video_codec = cv2.VideoWriter_fourcc(*'mp4v')
output_video = cv2.VideoWriter('output_video.mp4', video_codec, video_capture.get(cv2.CAP_PROP_FPS), 
                               (int(video_capture.get(3)), int(video_capture.get(4))))


# Обработка каждого кадра
for frame_index in range(len(all_player_coordinates)):
    # Получаем текущий кадр
    current_frame = all_images[frame_index].copy()
    
    # if frame_index % 5 == 0:
    #     # Отображаем текущий кадр
    #     plt.imshow(cv2.cvtColor(current_frame, cv2.COLOR_BGR2RGB))
    #     plt.show()

    # Накладываем изображение поля на текущий кадр
    current_frame[:field_image_height, :field_image_width] = field_image
    frame_height, frame_width, _ = current_frame.shape

    # # Получаем текущие результаты детекции
    # current_field_result = field_detection_results[frame_index]
    # current_player_result = player_detection_results[frame_index]

    # Извлечение классов объектов на поле
    # field_classes = current_field_result.boxes.cls.to('cpu').numpy()
    # field_boxes = current_field_result.boxes.xywhn.to('cpu').numpy()
    # field_masks = current_field_result.masks.xyn

    field_classes = all_field_classes[frame_index].copy()
    field_boxes = all_field_boxes[frame_index].copy()
    field_masks = all_field_masks[frame_index].copy()
    
    # Идентификация кругов и ворот на поле
    class_indices = {x:np.where(field_classes == y)[0] for x,y in class_to_id.items()}
    # center_indices = (field_classes == class_to_id['center'])
    # blue_line_indices = (field_classes == class_to_id['blue_line'])
    # circle_indices = (field_classes == class_to_id['red_circle'])
    # goal_indices = (field_classes == class_to_id['vorota'])

    # Координаты игроков на поле
    # player_coordinates = current_player_result.boxes.xyxyn.to('cpu').numpy()
    # players_x_center = player_coordinates[:, (0, 2)].sum(axis=1) / 2
    # players_y_bottom = player_coordinates[:, 3]
    # player_coordinates = np.column_stack((players_x_center, players_y_bottom))

    # player_coordinates = current_player_result.boxes.xywhn.to('cpu').numpy()
    player_cls = all_player_classes[frame_index]
    player_coordinates = all_player_coordinates[frame_index].copy()
    player_coordinates[:, 1] += player_coordinates[:, 3] / 2
    player_coordinates[:, 1] -= player_coordinates[:, 3] / 15
    player_coordinates = player_coordinates[:, :2]

    found_points = dict()

    transformed_player_coordinates = None
    
    if any(class_indices['center']):
        center_boxes = field_boxes[class_indices['center']]

        center_box = find_center(center_boxes)

        if center_box is not None:
            found_points['center'] = center_box
            # current_frame = cv2.line(current_frame, (int(center_box[0]*frame_width), 0), (int(center_box[0]*frame_width), frame_height), color=(0,255,0), thickness=3)

    
    map_coordinates = {
        'left_blue_line': 0,
        'right_blue_line': 1,
        'center': None,
        'left_circles': [],
        'right_circes': [],
    }

    if any(class_indices['blue_line']):

        # TODO: написать изменнеие границы при опускании вниз

        for idx in class_indices['blue_line']:
            cur_mask = field_masks[idx]
            cur_box = field_boxes[idx]

            highest_point = cur_mask[np.argmin(cur_mask[:, 1])]
            lowest_point = cur_mask[np.argmax(cur_mask[:, 1])]

            # current_frame = cv2.circle(current_frame, (int(highest_point[0]*frame_width), int(highest_point[1]*frame_height)), radius=10,color=(0,0,255),thickness=-1)

            if abs(lowest_point[0] - highest_point[0]) > 0.05:
                if lowest_point[0] > highest_point[0]:
                    map_coordinates['right_blue_line'] = lowest_point[0]
                    found_points['right_blue_line'] = lowest_point

                    if 0.01<highest_point[0]<0.99 and 0.01<highest_point[1]<0.99:
                        found_points['right_blue_line_up'] = highest_point

                    if 0.01<lowest_point[0]<0.99 and 0.01<lowest_point[1]<0.99:
                        found_points['right_blue_line_bottom'] = lowest_point
                    
                    color = (0, 0, 255)
                else:
                    map_coordinates['left_blue_line'] = lowest_point[0]
                    found_points['left_blue_line'] = lowest_point

                    if 0.01<highest_point[0]<0.99 and 0.01<highest_point[1]<0.99:
                        found_points['left_blue_line_up'] = highest_point

                    if 0.01<lowest_point[0]<0.99 and lowest_point[1]<0.99:
                        found_points['left_blue_line_bottom'] = lowest_point

                    color = (255, 0, 0)
                # current_frame = cv2.line(current_frame, (int(lowest_point[0]*frame_width), 0), (int(lowest_point[0]*frame_width), frame_height), color=color, thickness=3)

    if map_coordinates['left_blue_line'] == 0 and map_coordinates['right_blue_line'] == 1:
        if len(class_indices['vorota']) == 1 and len(class_indices['big_red_circle']):
            vorota_x = field_boxes[class_indices['vorota']][0][0]
            circle_mean_x = field_boxes[class_indices['big_red_circle']][:, 0].mean()
            if abs(vorota_x - circle_mean_x) > 0.05:
                if circle_mean_x > vorota_x:
                    map_coordinates['left_blue_line'] = 1
                else:
                    map_coordinates['right_blue_line'] = 0

    for circle_id in class_indices['red_circle']:
        circle_box = field_boxes[circle_id]
        
        if circle_box[0] < map_coordinates['left_blue_line']:
            if circle_box[1] > 0.58:

                if 'left_bottom_red_circle' not in found_points:
                    found_points['left_bottom_red_circle'] = circle_box
                else:
                    if found_points['left_bottom_red_circle'][1] > circle_box[1]:
                        found_points['left_top_red_circle'] = circle_box
                    else:
                        found_points['left_top_red_circle'] = found_points['left_bottom_red_circle']
                        found_points['left_bottom_red_circle'] = circle_box
            else:
                found_points['left_top_red_circle'] = circle_box
            # TODO: расписать куда что идет более подробно, не 1 if, учитывать ворота,другие точки
        elif circle_box[0] > map_coordinates['right_blue_line']:
            if circle_box[1] > 0.6:
                if 'right_bottom_red_circle' not in found_points:
                    found_points['right_bottom_red_circle'] = circle_box
                else:
                    if found_points['right_bottom_red_circle'][1] > circle_box[1]:
                        found_points['right_top_red_circle'] = circle_box
                    else:
                        found_points['right_top_red_circle'] = found_points['right_bottom_red_circle']
                        found_points['right_bottom_red_circle'] = circle_box
            else:
                found_points['right_top_red_circle'] = circle_box
            # TODO: расписать куда что идет более подробно, не 1 if, учитывать ворота,другие точки
        elif 'center' in found_points and circle_box[0] < found_points['center'][0]:
            if circle_box[1] > 0.6:
                found_points['center_left_bottom_red_circle'] = circle_box
            else:
                found_points['center_left_top_red_circle'] = circle_box
        elif 'center' in found_points and circle_box[0] > found_points['center'][0]:
            if circle_box[1] > 0.6:
                found_points['center_right_bottom_red_circle'] = circle_box
            else:
                found_points['center_right_top_red_circle'] = circle_box

    for vorota_id in class_indices['vorota']:
        vorota_box = field_boxes[vorota_id]
        vorota_box[1] += vorota_box[3] / 4
        if vorota_box[0] < map_coordinates['left_blue_line']:
            vorota_box[0] += vorota_box[2] / 10
            found_points['left_vorota'] = vorota_box
        elif vorota_box[0] > map_coordinates['right_blue_line']:
            vorota_box[0] -= vorota_box[2] / 10
            found_points['right_vorota'] = vorota_box

    for big_circle_id in class_indices['big_red_circle']:
        big_circle_box = field_boxes[big_circle_id]
        big_circle_mask = field_masks[big_circle_id]

        if big_circle_box[0] < map_coordinates['left_blue_line']:
            right_point = big_circle_mask[np.argmax(big_circle_mask[:, 0])]
            right_point[1] = big_circle_box[1]
            if big_circle_box[1] > 0.6:
                found_points['left_bottom_red_circle_upper_point'] = right_point
            else:
                found_points['left_top_red_circle_upper_point'] = right_point

        elif big_circle_box[0] > map_coordinates['right_blue_line']:
            left_point = big_circle_mask[np.argmin(big_circle_mask[:, 0])]
            left_point[1] = big_circle_box[1]
            if big_circle_box[1] > 0.6:
                found_points['right_bottom_red_circle_upper_point'] = left_point
            else:
                found_points['right_top_red_circle_upper_point'] = left_point

    image_points = []
    real_points = []
    for name, point in found_points.items():
        if 'blue_line' in name and 'up' not in name and 'bottom' not in name:
            continue

        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1
        color = (255, 0, 0)
        thickness = 2

        cv2.putText(current_frame, str(field_numbers[name]), (int(point[0]*frame_width), int(point[1]*frame_height)), font, font_scale, color, thickness)
        current_frame = cv2.circle(current_frame, 
                                    center=(int(point[0]*frame_width), int(point[1]*frame_height)), 
                                    radius=3, color=(0,255,0), thickness=-1)


        image_points.append([point[0], point[1]])
        real_points.append(real_field_coordinates[name])


    # if len(real_points) == 3:
    #     affine_transform_matrix = cv2.getAffineTransform(np.float32(image_points), np.float32(real_points))
    #     transformed_player_coordinates = cv2.transform(np.float32([player_coordinates]), affine_transform_matrix)[0]
    if len(real_points) > 3:
        homography_matrix, _ = cv2.findHomography(np.float32(image_points), np.float32(real_points))
        transformed_player_coordinates = cv2.perspectiveTransform(np.float32([player_coordinates]), homography_matrix)[0]

    if transformed_player_coordinates is not None and (transformed_player_coordinates.max() > 1.2 or transformed_player_coordinates.min() < -0.2):
        # Какое-то гавно при трансформации - не показывать игроков
        transformed_player_coordinates = None

    if transformed_player_coordinates is not None:
        for (player_x, player_y), cls in zip(transformed_player_coordinates, player_cls):
            color = [(180,133,66),(150,0,66)][int(cls)]
            current_frame = cv2.circle(current_frame, 
                                    center=(int(player_x * field_image_width), int(player_y * field_image_height)), 
                                    radius=5, color=color, thickness=-1)
        
        transformed_player_coordinates = None

    for (x,y), cls in zip(player_coordinates, player_cls):
        # current_frame = cv2.circle(current_frame, 
        #                         center=(int(x*frame_width), int(y*frame_height)), 
        #                         radius=5, color=(0, 0, 255), thickness=-1)
        overlay = current_frame.copy()
        color = [(180,133,66),(150,0,66)][int(cls)]
        overlay = cv2.ellipse(overlay, (int(x*frame_width), int(y*frame_height)), (40, 16), 0, -40, 220, color, 8)

        alpha = 0.5  # Прозрачность
        current_frame = cv2.addWeighted(overlay, alpha, current_frame, 1 - alpha, 0, current_frame)

    # current_frame = cv2.line(current_frame, (0, int(frame_height * 0.6)), (frame_width,int(frame_height*0.6)),color=(0,255,0),thickness=3)

    output_video.write(current_frame)

# Закрытие выходного видео
output_video.release()

del output_video